[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/graphene/graphene.ipynb)

# Electrons in Graphene

This example computes the electronic bands of graphene and the Berry phase of its electrons around the Brillouin-zone corners. Graphene is modelled as a honeycomb sheet of carbon atoms whose two sublattices couple through nearest-neighbour hopping; near the corners where its two bands meet in cones, electrons move at a constant speed of about a million metres per second, while breaking sublattice symmetry as in boron nitride opens a band gap. An electron's state records how its amplitude is shared between the two sublattices as a pseudospin, and carrying the state once around a gapless cone flips its sign, giving the half-turn Berry phase behind graphene's half-integer quantum Hall plateaus and suppressed backscattering.

The electron state is modelled as an even multivector spinor in three-dimensional space. Hopping sums the plane `yz` turned in `xy` by each bond phase `momentum & bonds` into an extrinsic bivector pseudospin field, while a state's pseudospin sandwiches the intrinsic sublattice patch `(upper_states >> first_sublattice) / upper_states.symmetric_reverse_product()`. The Hamiltonian `-(Field * Spinor * xy)` squares to the field's squared norm, giving band energies at plus and minus the field length, and multiplying the minimal step rotations `(1 + directions[1:] * directions[:-1]).normalized()` around a momentum loop yields the holonomy rotor holding the Berry phase.

In tight-binding matrix notation, an electron state is written as a two-component complex column $u(\mathbf k) = (u_A, u_B)^\mathsf{T}$, each nearest-neighbour vector $\boldsymbol\delta_j$ contributes the complex phase $e^{i\mathbf k\cdot\boldsymbol\delta_j}$ to the off-diagonal hopping sum $f(\mathbf k) = -t\sum_{j=1}^3 e^{i\mathbf k\cdot\boldsymbol\delta_j}$, and the $2\times 2$ Bloch Hamiltonian $H(\mathbf k) = \begin{pmatrix} m & f(\mathbf k) \\ f^*(\mathbf k) & -m \end{pmatrix} = \mathbf d(\mathbf k)\cdot\boldsymbol\sigma$ requires gauge-dependent complex eigenvectors around a loop so the Berry phase can be extracted from $\operatorname{Im}\ln\prod_i \langle u(\mathbf k_{i+1})|u(\mathbf k_i)\rangle$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from functools import partial

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure
from examples.quantum.graphene import render

np.set_printoptions(precision=3, suppress=True)

# The geometric algebra of three-dimensional space.
ga = VGA3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Scalar = ga.gatype.scalar()                                                     # Scalar+
# A momentum, read by pairing with a displacement, is a wavevector.
Wavevector = ga.gatype.vector()                                                 # Vector+
Displacement = ga.gatype.antivector()                                           # Antivector-
# The field that turns a pseudospin, and a pseudospin.
Field = ga.gatype.bivector()                                                    # Bivector+
Pseudospin = ga.gatype.antibivector()                                           # Antibivector-
# The state of an electron.
Spinor = ga.gatype.even()                                                       # Even+
Rotor = ga.gatype.rotor()                                                       # Rotor+
# The energy of a hop between neighbouring atoms, in eV.
hopping = 2.8
# The bonds from an atom to its three neighbours, a third of a turn apart, in carbon-carbon distances: displacements,
# antivectors.
bond_angles = np.array([1, 3, 5]) * np.pi / 3
bond_turns = (mv.xy * (-bond_angles / 2)).exp()                                 # [3] Rotor+
bonds = bond_turns >> mv.x.dual()                                               # [3] Displacement-
# The two inequivalent corners of the Brillouin zone, K and K'.
valleys = mv.vector([[2 * np.pi / 3, 2 * np.pi / (3 * np.sqrt(3)), 0],
                     [2 * np.pi / 3, -2 * np.pi / (3 * np.sqrt(3)), 0]])        # [2] Wavevector+

## Pseudospin field

An electron in graphene hops between neighbouring atoms, and every neighbour of an atom belongs to the other sublattice. Each bond contributes the plane `yz` turned in the plane `xy` by the phase `momentum & bonds` that the wavevector picks up across that bond, summing over the three bonds into one extrinsic bivector field at each momentum; when the two sublattices differ in energy, their difference adds a gap in `xy`.

In [ ]:
# Momenta are in inverse carbon-carbon distances (1 / 0.142 nm), covering the Brillouin zone without a gap.
zone_extent = 4.5
n_zone_samples = 121
gapless = np.array(0.0)

In [ ]:
def pseudospin_field(momentum: Wavevector, gap: np.ndarray) -> Field:
    """Hopping across the three bonds turns the plane `yz` in `xy` by each bond phase and adds the sublattice gap in `xy` (`[...] Field+`)."""
    # The phase across each bond: the momentum, a wavevector and so a plane, paired with the bond.
    phases = momentum[..., None] & bonds                                        # [..., 3] Scalar+
    # The plane yz turned in the plane xy by each phase, summed over the bonds, plus the gap in the plane xy.
    return -hopping * ((mv.xy * phases).exp() * mv.yz).sum(axis=-1) + gap * mv.xy   # [...] Field+


# Momenta on a square that holds the Brillouin zone.
zone_axis = np.linspace(-zone_extent, zone_extent, n_zone_samples)
kx, ky = np.meshgrid(zone_axis, zone_axis)
momenta = mv.x * kx + mv.y * ky                                                 # [n_zone_samples, n_zone_samples] Wavevector+
zone_field = pseudospin_field(momenta, gapless)                                 # [n_zone_samples, n_zone_samples] Field+
save_figure(render.field_figure(momenta, zone_field), "graphene_field");

Brightness shows the field's length, peaking at three times the hopping at the zone centre and vanishing at the six Brillouin-zone corners where the three bond phases sit a third of a turn apart, while the white streaks trace the in-plane direction winding once around each corner in opposite senses at the two inequivalent valleys.

## Hamiltonian and electronic bands

The Hamiltonian `-(Field * Spinor * xy)` multiplies a spinor by minus the pseudospin field on the left and the plane `xy` on the right. Applied twice at one momentum, both bivectors square to minus their squared norms and multiply every state by the field's squared length, so the two bands sit at plus and minus the field length and meet in linear cones of slope one and a half times the hopping at the six valleys.

In Pauli-matrix notation, the state reads as a two-component complex column and the Hamiltonian as the $2\times 2$ matrix $\mathbf d\cdot\boldsymbol\sigma$, with $\mathbf d$ collecting the real and imaginary parts of $f(\mathbf k)$ alongside the sublattice mass, where applying the operator twice reads as $(\mathbf d\cdot\boldsymbol\sigma)^2 = |\mathbf d|^2\,\mathbb 1$.

In [ ]:
# The Hamiltonian, bound to a field a map on spinors: minus the field on the left, the plane xy on the right.
# Applied twice the field meets itself and the plane xy meets itself, each squaring to minus its squared length,
# so the energies are plus and minus the field's length.
hamiltonian = -(Field * Spinor * mv.xy)                                         # [] Spinor+ <- (Field+, Spinor+)
energies, _ = hamiltonian(zone_field).eigh()                                    # [n_zone_samples, n_zone_samples, 4] Scalar+
save_figure(render.bands_figure(momenta, energies), "graphene_bands");

The red upper sheet and blue lower sheet touch at the six Dirac cones around the Brillouin-zone perimeter and spread apart toward the zone centre.

## Pseudospin around the valleys

Sandwiched with an electron spinor and normalized by `psi.symmetric_reverse_product()`, the first-sublattice patch `first_sublattice` gives the state's unit intrinsic pseudospin; in the upper band it aligns with `field.dual()`. Without a gap it stays in the plane and winds once around each valley, whereas a sublattice gap tilts the pseudospin out of the plane at the valley centres where only the `xy` gap component remains.

In Pauli-matrix notation, a two-component eigenvector $u = (u_A, u_B)^\mathsf{T}$ gives the pseudospin expectation $\langle u|\boldsymbol\sigma|u\rangle / \langle u|u\rangle$, splitting into the sublattice imbalance $(|u_A|^2 - |u_B|^2)/\langle u|u\rangle$ out of the plane and the complex coherence $2\,u_A^* u_B/\langle u|u\rangle$ in the plane.

In [ ]:
# Momentum square half-width, sample count, and sublattice gap in eV around each valley.
valley_extent = 0.5
n_valley_samples = 11
texture_gap = np.array(0.3)

In [ ]:
# The pseudospin of a state wholly on the first sublattice: the patch of the plane xy.
first_sublattice = mv.xy.dual()                                                 # [] Pseudospin-


def upper_pseudospins(offsets: Wavevector, gap: np.ndarray) -> Pseudospin:
    """Diagonalizing the Hamiltonian near both valleys gives the upper band's unit pseudospin (`[2, ...] Pseudospin-`)."""
    valley_field = pseudospin_field(valleys[:, None, None] + offsets, gap)      # [2, ...] Field+
    _, states = hamiltonian(valley_field).eigh()                                # [2, ..., 4] Spinor+
    upper_states = states[..., -1]                                              # [2, ...] Spinor+
    # The pseudospin of a state: its sandwich of the first sublattice's pseudospin, over its norm squared.
    return (upper_states >> first_sublattice) / upper_states.symmetric_reverse_product()   # [2, ...] Pseudospin-


# Momenta on a small square about each valley.
valley_axis = np.linspace(-valley_extent, valley_extent, n_valley_samples)
kx, ky = np.meshgrid(valley_axis, valley_axis)
offsets = mv.x * kx + mv.y * ky                                                 # [n_valley_samples, n_valley_samples] Wavevector+
valley_directions = upper_pseudospins(offsets, texture_gap)                     # [2, n_valley_samples, n_valley_samples] Pseudospin-
save_figure(render.textures_figure(partial(upper_pseudospins, gap=texture_gap), valley_extent), "graphene_pseudospin");

Red shading marks pseudospin tilted toward positive out-of-plane `z` at both valley centres by the sublattice gap, while the dark streaks wind once in opposite senses around the first valley on the left and the second valley on the right.

## Berry phase around a valley

Carrying a frame along the upper-band pseudospin around a closed momentum loop by the minimal step rotors `(1 + directions[1:] * directions[:-1]).normalized()` brings the frame back turned about its starting direction by the solid angle enclosed on the pseudospin sphere. Their cumulative product gives the holonomy rotor, whose scalar part is the cosine of the Berry phase: without a gap the pseudospin traces the equator for a half-turn Berry phase and a rotor of `-1`, while a gap lifts small loops toward the pole so the phase rises smoothly from zero toward pi.

In tight-binding eigenvector notation, the discrete Berry phase is computed from the complex Bargmann product $\gamma = \arg\prod_i \langle u(\mathbf k_{i+1}) | u(\mathbf k_i)\rangle$, or in the smooth gauge limit as the line integral $\gamma = \oint i\langle u | \nabla_{\mathbf k} u\rangle \cdot d\mathbf k$ of a gauge-dependent Berry connection around the loop.

In [ ]:
# Step count around each loop, loop radii in inverse carbon-carbon distances, and sublattice gaps in eV.
n_steps = 400
n_radii = 30
radii = np.linspace(0.01, 0.6, n_radii)
n_gaps = 4
gaps = np.array([0.0, 0.2, 0.5, 1.0])

In [ ]:
def transport(directions: Pseudospin) -> Rotor:
    """Minimal step rotations along a curve of unit pseudospins accumulate into the frame transport rotors (`[...] Rotor+`), ending at the loop holonomy."""
    # The smallest rotation from each direction to the next.
    step_rotors = (1 + directions[1:] * directions[:-1]).normalized()           # [...] Rotor+
    # Multiplied up in order, each step after the ones before.
    return step_rotors.cumprod(axis=0)                                          # [...] Rotor+


def momentum_circle(n_steps: int) -> Wavevector:
    """Steps a unit wavevector around the momentum circle in `n_steps` intervals (`[n_steps + 1] Wavevector+`), closing at the start."""
    angles = np.linspace(0.0, 2 * np.pi, n_steps + 1)
    turns = (mv.xy * (-angles / 2)).exp()                                       # [n_steps + 1] Rotor+
    return turns >> mv.x                                                        # [n_steps + 1] Wavevector+


# Circles about each valley, for every radius and gap at once.
circle = momentum_circle(n_steps)                                               # [n_steps + 1] Wavevector+
loops = valleys[:, None] + circle[:, None, None] * radii                        # [n_steps + 1, 2, n_radii] Wavevector+
loop_field = pseudospin_field(loops[:, None], gaps[:, None, None])              # [n_steps + 1, n_gaps, 2, n_radii] Field+
# The upper band's pseudospin, along the field.
directions = loop_field.normalized().dual()                                     # [n_steps + 1, n_gaps, 2, n_radii] Pseudospin-
holonomies = transport(directions)[-1]                                          # [n_gaps, 2, n_radii] Rotor+
save_figure(render.berry_figure(radii, gaps, holonomies, directions[0]), "graphene_berry");

Solid curves show the first valley and dashed curves the second valley for each sublattice gap: gapless loops sit flat at pi, while gapped loops start near zero at small radius and approach minus pi in the first valley and plus pi in the second as the loop outgrows the gap.

In [ ]:
# A small loop sees a cone: its Berry phase is pi times one less the gap over the field's length on the loop,
# with the field growing at the cones' slope away from the valley.
cone_slope = 1.5 * hopping
cone_phases = np.pi * (1 - gaps / np.sqrt(gaps**2 + (cone_slope * radii[0]) ** 2))
# Each holonomy's angle, signed by its sense about the pseudospin direction the loop starts from: a rotor turning a
# positive angle about the start has minus the start's plane in its bivector part.
sine_parts = (-holonomies.select[2]) & directions[0]                            # [n_gaps, 2, n_radii] Scalar+
print("smallest loop, Berry phase / pi, K and K':\n", np.arctan2(sine_parts.to_array(), holonomies.select[0].to_array())[:, :, 0] / np.pi)
print("cone Berry phase / pi:", cone_phases / np.pi)

In [ ]:
# checks
# The energies are minus and plus the field's length, and the field vanishes at the valleys; the upper
# band's pseudospin points along the field; without a gap every loop comes back turned by a full turn,
# and a small loop with a gap has the cone's phase, compared through one minus its cosine so that
# small phases stay visible.
length = zone_field.norm()                                                      # [n_zone_samples, n_zone_samples] Scalar+
expected_energies = stack((-length, -length, length, length), axis=-1)          # [n_zone_samples, n_zone_samples, 4] Scalar+
assert ((energies - expected_energies).abs() < 1e-9).all()
assert (pseudospin_field(valleys, gapless).norm() < 1e-6).all()
valley_field = pseudospin_field(valleys[:, None, None] + offsets, texture_gap)  # [2, n_valley_samples, n_valley_samples] Field+
assert ((valley_directions - valley_field.normalized().dual()).antinorm() < 1e-9).all()
assert ((holonomies[gaps == 0.0].select[0] + 1).abs() < 1e-9).all()
expected_drop = 1 - np.cos(cone_phases)[:, None]
actual_drop = 1 - holonomies[:, :, 0].select[0]                                 # [n_gaps, 2] Scalar+
assert ((actual_drop - expected_drop).abs() <= 1e-2 * expected_drop).all()